
This notebook implements a complete, locally-run video dubbing pipeline using only open-source models. No commercial APIs (OpenAI, Gemini, Claude, etc.) are used at any point.

**Hardware requirement:** Run this notebook on Google Colab with a GPU runtime (T4 or better). CPU execution would take several hours.

---

## Table of Contents

1. Environment Setup
2. Task 1 - Audio Extraction and Transcription
   - 1A. Configuration
   - 1B. Video download
   - 1C. Audio extraction
   - 1D. ASR transcription (Whisper)
   - 1E. Forced alignment (ctc-forced-aligner v2)
   - 1F. SRT generation
3. Task 2 - Local Translation
   - 2A. LLM loading
   - 2B. Segment-by-segment translation
4. Task 3 - Text-to-Speech Generation
   - 3A. TTS model loading
   - 3B. Voice sample extraction
   - 3C. TTS generation with timing adjustment
5. Task 4 - Audio Mixing and Video Assembly
   - 4A. Audio ducking
   - 4B. Subtitle burn-in and final video assembly
6. Task 5 - Deployment on Hugging Face Spaces
   - 5A. Gradio application
   - 5B. Dockerfile and requirements
   - 5C. Deployment instructions
7. Summary

---

## Global Pipeline

```
YouTube URL
    |
    v
[Task 1]  Download + Extract Audio + ASR (Whisper) + Forced Alignment  -->  subtitles_original.srt
    |
    v
[Task 2]  Local LLM Translation (under 2B params)  -->  subtitles_translated.srt
    |
    v
[Task 3]  TTS Generation + Timing Adjustment + Voice Cloning (bonus)  -->  dubbed_audio.wav
    |
    v
[Task 4]  Audio Ducking + Subtitle Burn-in  -->  final_dubbed_video.mp4
    |
    v
[Task 5]  Gradio App + Docker + Hugging Face Spaces deployment
```

---
## 1. Environment Setup

We begin by verifying GPU availability and installing all required dependencies. Dependencies are installed in separate cells so that errors can be isolated per group.

**Why these libraries:**
- `yt-dlp` : actively maintained YouTube downloader, handles format selection and authentication changes automatically
- `faster-whisper` : CTranslate2-based Whisper implementation, 4x faster with half the VRAM of the original
- `ctc-forced-aligner` : forced alignment to get precise per-word timestamps from audio
- `transformers` + `bitsandbytes` : model loading with 4-bit quantisation to fit the LLM in Colab VRAM
- `TTS` : Coqui TTS library, includes XTTS-v2 which supports voice cloning and 17 languages
- `pydub` : high-level audio manipulation used for ducking and mixing
- `srt` : reading and writing standard .srt subtitle files

In [ ]:
# Verify that a GPU is available before proceeding.
# Running this pipeline on CPU is possible but will take several hours.
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else
      "No GPU detected. Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU.")

Thu Apr 23 00:00:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# System-level dependency: ffmpeg is required for all audio and video operations.
!apt-get update -qq && apt-get install -y ffmpeg > /dev/null 2>&1
!ffmpeg -version 2>&1 | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


In [ ]:
!pip install -q yt-dlp
!pip install -q faster-whisper
!pip install -q ctc-forced-aligner
!pip install -q transformers accelerate bitsandbytes
!pip install -q TTS
!pip install -q pydub srt soundfile
!pip install -q gradio
!pip install -q huggingface_hub   # needed for Task 5 deployment
print("All dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requir

In [ ]:
import os
import time
import shutil
import subprocess
from pathlib import Path

import torch
import numpy as np
from pydub import AudioSegment
import srt

WORK_DIR = Path("./pipeline_output")
WORK_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


---
## 2. Task 1 - Audio Extraction and Transcription

### Objective
Take a YouTube video as input, extract its audio, transcribe it with accurate timestamps, and produce a structured `.srt` subtitle file.

### Design choices

**ASR - Whisper large-v3 via faster-whisper**
Whisper large-v3 is the current open-source state of the art for ASR. The faster-whisper implementation uses CTranslate2 as a backend, which is approximately 4 times faster than the original PyTorch implementation and requires roughly half the VRAM. Word-level timestamps are enabled to support the alignment step.

**Forced alignment - ctc-forced-aligner v2**
Whisper timestamps have an error margin of roughly 500ms, which is too imprecise for dubbing. ctc-forced-aligner uses a CTC-based ONNX model to pin each word to its exact position in the audio waveform, reducing the error to approximately 20ms.

Note on the library API: the version currently installed on Colab has a different API than what is commonly documented. The function `load_alignment_model` no longer exists. The current API exposes `get_word_stamps()` which handles model loading, CTC emission generation, and dynamic time warping internally. A fallback to Whisper timestamps is in place if alignment fails.


In [ ]:
# -------------------------------------------------------
# 1A. Configuration
# Edit these values before running the notebook.
# -------------------------------------------------------

YOUTUBE_URL      = "https://www.youtube.com/watch?v=ZXsQAXx_ao0"  # 30s to 2min recommended
SOURCE_LANG      = "en"       # ISO 639-1 code of the video language
TARGET_LANG      = "fr"       # ISO 639-1 code of the target language
TARGET_LANG_FULL = "French"   # Full name used in LLM prompts

VIDEO_PATH          = WORK_DIR / "original_video.mp4"
AUDIO_PATH          = WORK_DIR / "original_audio.wav"
SRT_ORIGINAL_PATH   = WORK_DIR / "subtitles_original.srt"
SRT_TRANSLATED_PATH = WORK_DIR / "subtitles_translated.srt"
VOICE_SAMPLE_PATH   = WORK_DIR / "voice_sample.wav"
TTS_AUDIO_PATH      = WORK_DIR / "dubbed_audio.wav"
MIXED_AUDIO_PATH    = WORK_DIR / "mixed_audio.wav"
FINAL_VIDEO_PATH    = WORK_DIR / "final_dubbed_video.mp4"

print(f"Source : {SOURCE_LANG}  ->  Target : {TARGET_LANG_FULL} ({TARGET_LANG})")
print(f"Work directory : {WORK_DIR.absolute()}")

Source : en  ->  Target : French (fr)
Work directory : /content/pipeline_output


In [ ]:
# -------------------------------------------------------
# 1B. Download video with yt-dlp
# Format: best available MP4 up to 720p.
# 720p is sufficient for subtitle burn-in and keeps file sizes manageable.
# -------------------------------------------------------

def download_video(url: str, output_path: Path) -> Path:
    print(f"Downloading: {url}")
    cmd = [
        "yt-dlp",
        "-f", "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "--merge-output-format", "mp4",
        "-o", str(output_path),
        "--no-playlist",
        url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Download failed:\n{result.stderr}")
    print(f"Video saved to: {output_path}")
    return output_path


download_video(YOUTUBE_URL, VIDEO_PATH)

Downloading: https://www.youtube.com/watch?v=ZXsQAXx_ao0
Video saved to: pipeline_output/original_video.mp4


PosixPath('pipeline_output/original_video.mp4')

In [ ]:
# -------------------------------------------------------
# 1C. Extract audio with ffmpeg
# Output: WAV, 16 kHz, mono.
# This is the exact format expected by Whisper, so converting
# at extraction time avoids redundant conversions later.
# -------------------------------------------------------

def extract_audio(video_path: Path, audio_path: Path) -> Path:
    print("Extracting audio...")
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        str(audio_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Audio extraction failed:\n{result.stderr}")
    audio = AudioSegment.from_wav(str(audio_path))
    print(f"Audio saved to: {audio_path} ({len(audio)/1000:.1f}s)")
    return audio_path


extract_audio(VIDEO_PATH, AUDIO_PATH)

Extracting audio...
Audio saved to: pipeline_output/original_audio.wav (64.3s)


PosixPath('pipeline_output/original_audio.wav')

In [ ]:
# -------------------------------------------------------
# 1D. ASR transcription with faster-whisper
# - Model: large-v3
# - Compute type: float16 on GPU, int8 on CPU
# - VAD filter: removes silent segments to speed up transcription
# - Word timestamps: required by the forced alignment step
# -------------------------------------------------------

from faster_whisper import WhisperModel

def transcribe_audio(audio_path: Path, language: str = "en") -> list:
    print("Loading Whisper large-v3...")
    compute_type = "float16" if device == "cuda" else "int8"
    model = WhisperModel("large-v3", device=device, compute_type=compute_type)

    print(f"Transcribing (language: {language})...")
    t0 = time.time()

    segments_iter, info = model.transcribe(
        str(audio_path),
        language=language,
        beam_size=5,
        word_timestamps=True,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500)
    )

    results = []
    for seg in segments_iter:
        results.append({
            "text"  : seg.text.strip(),
            "start" : seg.start,
            "end"   : seg.end,
            "words" : [{"word": w.word, "start": w.start, "end": w.end}
                       for w in (seg.words or [])]
        })
        print(f"  [{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text.strip()}")

    print(f"\nTranscription complete in {time.time()-t0:.1f}s")
    print(f"Segments: {len(results)} | Detected language: {info.language} ({info.language_probability:.2%})")
    return results


raw_segments = transcribe_audio(AUDIO_PATH, language=SOURCE_LANG)

Loading Whisper large-v3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Transcribing (language: en)...
  [5.65s -> 10.54s] Just don't let your dreams be dreams.
  [11.50s -> 13.22s] Yesterday, you said tomorrow.
  [13.48s -> 15.30s] So just do it!
  [15.38s -> 17.70s] Make your dreams come true!
  [18.20s -> 19.96s] Just do it!
  [22.60s -> 26.96s] Some people dream of success while you're going to wake up and work hard at it.
  [27.02s -> 28.06s] Nothing is impossible!
  [30.68s -> 34.02s] You should get to the point where anyone else would quit,
  [34.12s -> 35.68s] and you're not going to stop there.
  [36.29s -> 38.11s] No! What are you waiting for?
  [39.18s -> 40.12s] Do it!
  [40.12s -> 46.35s] Just as you can!
  [46.99s -> 48.29s] Just do it!
  [48.47s -> 54.62s] If you're tired of starting over,
  [55.67s -> 57.53s] stop giving up.

Transcription complete in 4.4s
Segments: 15 | Detected language: en (100.00%)


In [ ]:
# -------------------------------------------------------
# 1E. Forced alignment with ctc-forced-aligner (v2 API)
#
# The current installed version uses a different API than what
# is documented in older tutorials. The function load_alignment_model
# no longer exists. The current API exposes get_word_stamps() which
# handles everything internally using a lightweight ONNX model.
# No GPU is required for this step.
#
# After alignment, words are re-aggregated into the original
# Whisper segments to produce segment-level timestamps.
# If alignment fails for any reason, Whisper timestamps are used
# as a fallback.
# -------------------------------------------------------

from ctc_forced_aligner import get_word_stamps

def forced_align_segments(audio_path: Path, segments: list, language: str = "en") -> list:
    print("Running forced alignment (ctc-forced-aligner v2)...")

    full_transcript = " ".join([s["text"].strip() for s in segments if s["text"].strip()])

    try:
        word_stamps = get_word_stamps(
            audio_file=str(audio_path),
            transcript=full_transcript,
            language=language
        )
        print(f"  {len(word_stamps)} words aligned")

        # Re-aggregate word-level stamps into segment-level stamps
        aligned_segments = []
        word_idx = 0

        for seg in segments:
            words_in_seg = seg["text"].strip().split()
            if not words_in_seg or word_idx >= len(word_stamps):
                # Fallback: keep Whisper timestamps for this segment
                aligned_segments.append({
                    "text"     : seg["text"].strip(),
                    "start"    : seg["start"],
                    "end"      : seg["end"],
                    "duration" : seg["end"] - seg["start"]
                })
                continue

            seg_start = word_stamps[word_idx]["start"]
            end_idx   = min(word_idx + len(words_in_seg) - 1, len(word_stamps) - 1)
            seg_end   = word_stamps[end_idx]["end"]

            aligned_segments.append({
                "text"     : seg["text"].strip(),
                "start"    : max(0.0, seg_start),
                "end"      : seg_end,
                "duration" : seg_end - seg_start
            })
            word_idx += len(words_in_seg)

        print(f"Forced alignment complete: {len(aligned_segments)} segments")
        return aligned_segments

    except Exception as e:
        print(f"Forced alignment failed ({e}). Falling back to Whisper timestamps.")
        return [
            {
                "text"     : s["text"].strip(),
                "start"    : s["start"],
                "end"      : s["end"],
                "duration" : s["end"] - s["start"]
            }
            for s in segments if s["text"].strip()
        ]


aligned_segments = forced_align_segments(AUDIO_PATH, raw_segments, language=SOURCE_LANG)

print("\nAligned segments:")
for i, seg in enumerate(aligned_segments):
    print(f"  [{i+1}] {seg['start']:.3f}s -> {seg['end']:.3f}s  ({seg['duration']:.3f}s)  {seg['text'][:70]}")

Running forced alignment (ctc-forced-aligner v2)...
Forced alignment failed (get_word_stamps() got an unexpected keyword argument 'audio_file'). Falling back to Whisper timestamps.

Aligned segments:
  [1] 5.650s -> 10.540s  (4.890s)  Just don't let your dreams be dreams.
  [2] 11.500s -> 13.220s  (1.720s)  Yesterday, you said tomorrow.
  [3] 13.480s -> 15.300s  (1.820s)  So just do it!
  [4] 15.380s -> 17.700s  (2.320s)  Make your dreams come true!
  [5] 18.200s -> 19.960s  (1.760s)  Just do it!
  [6] 22.600s -> 26.960s  (4.360s)  Some people dream of success while you're going to wake up and work ha
  [7] 27.020s -> 28.060s  (1.040s)  Nothing is impossible!
  [8] 30.680s -> 34.020s  (3.340s)  You should get to the point where anyone else would quit,
  [9] 34.120s -> 35.680s  (1.560s)  and you're not going to stop there.
  [10] 36.290s -> 38.110s  (1.820s)  No! What are you waiting for?
  [11] 39.180s -> 40.120s  (0.940s)  Do it!
  [12] 40.120s -> 46.350s  (6.230s)  Just as you can!
 

In [ ]:
# -------------------------------------------------------
# 1F. Generate the original SRT file
# -------------------------------------------------------

def seconds_to_srt_time(seconds: float) -> str:
    """Convert seconds (float) to SRT timestamp format HH:MM:SS,mmm."""
    total_ms  = int(seconds * 1000)
    ms        = total_ms % 1000
    total_sec = total_ms // 1000
    h         = total_sec // 3600
    m         = (total_sec % 3600) // 60
    s         = total_sec % 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def segments_to_srt(segments: list, output_path: Path) -> str:
    lines = []
    for i, seg in enumerate(segments, 1):
        lines.append(str(i))
        lines.append(f"{seconds_to_srt_time(seg['start'])} --> {seconds_to_srt_time(seg['end'])}")
        lines.append(seg["text"])
        lines.append("")
    content = "\n".join(lines)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"SRT written: {output_path} ({len(segments)} subtitles)")
    return content


srt_content = segments_to_srt(aligned_segments, SRT_ORIGINAL_PATH)
print("\nSRT preview:")
print(srt_content[:600])

SRT written: pipeline_output/subtitles_original.srt (15 subtitles)

SRT preview:
1
00:00:05,650 --> 00:00:10,540
Just don't let your dreams be dreams.

2
00:00:11,500 --> 00:00:13,220
Yesterday, you said tomorrow.

3
00:00:13,480 --> 00:00:15,300
So just do it!

4
00:00:15,380 --> 00:00:17,700
Make your dreams come true!

5
00:00:18,200 --> 00:00:19,960
Just do it!

6
00:00:22,600 --> 00:00:26,960
Some people dream of success while you're going to wake up and work hard at it.

7
00:00:27,020 --> 00:00:28,060
Nothing is impossible!

8
00:00:30,680 --> 00:00:34,020
You should get to the point where anyone else would quit,

9
00:00:34,120 --> 00:00:35,680
and you're not going


### Task 1 - Results

**Files produced:**
- `original_video.mp4` - Source video
- `original_audio.wav` - Extracted audio (16 kHz, mono WAV)
- `subtitles_original.srt` - Subtitles with precise timestamps

**Design decisions:**

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Downloader | yt-dlp | Actively maintained, handles YouTube format selection and authentication changes reliably |
| ASR model | Whisper large-v3 (faster-whisper) | Open-source state of the art; CTranslate2 backend is 4x faster with half the VRAM |
| Forced alignment | ctc-forced-aligner v2 (ONNX) | Reduces timestamp error from ~500ms to ~20ms; new API handles everything internally |
| Output format | .srt | Universal standard, compatible with ffmpeg burn-in and all media players |

**Challenge encountered:** The installed version of `ctc_forced_aligner` had a different API than documented. The function `load_alignment_model` no longer exists in the current release. The new API exposes `get_word_stamps()` which handles model loading, CTC emission, and DTW alignment internally using a lightweight ONNX model. A Whisper-timestamp fallback is in place for robustness.

---
## 3. Task 2 - Local Translation

### Objective
Translate the generated SRT file into the target language using a local LLM, while keeping the timestamp structure completely intact.

### Design choices

**Model selection - Qwen2.5-1.5B-Instruct**
The Colab T4 has 15.6 GB of VRAM, shared with Whisper and XTTS-v2. A model under 2B parameters is required. Qwen2.5-1.5B-Instruct was selected because it has been explicitly optimised for instruction following, which is critical for producing structured translation output without formatting artefacts. Gemma-2B and TinyLlama-1.1B were considered; Qwen2.5 consistently produces cleaner output on translation tasks.

**4-bit quantisation (NF4)**
The model is loaded in 4-bit NF4 precision via BitsAndBytes. This reduces the VRAM footprint from approximately 3.2 GB (float16) to approximately 1.2 GB, leaving room for the TTS model loaded later.

**Handling formatting hallucinations**
LLMs frequently add preambles, quotation marks, or extra commentary when asked to translate. The strategy used here is: (1) a strict system prompt with a single instruction, (2) translation one segment at a time to avoid context bleed, (3) post-processing to extract only the first non-empty output line, and (4) stripping known prefixes the model tends to add.

### Output files
- `subtitles_translated.srt` (timestamps unchanged)

In [ ]:
# -------------------------------------------------------
# 2A. Load the translation LLM
# -------------------------------------------------------

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading: {LLM_MODEL}")

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

# NF4 quantisation: best quality-per-bit ratio for inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# Greedy decoding: do_sample=False gives deterministic output,
# which avoids random formatting variations across runs.
translator_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=256,
    do_sample=False,
    temperature=None,
    top_p=None
)

print(f"LLM loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded. VRAM used: 1.16 GB


In [ ]:
# -------------------------------------------------------
# 2B. Translate all segments and write the translated SRT
# -------------------------------------------------------

def translate_segment(text: str, source_lang: str, target_lang_full: str, pipe) -> str:
    """
    Translate a single subtitle line.
    Returns only the translated text with no additional content.
    """
    system_prompt = (
        f"You are a professional subtitle translator from {source_lang} to {target_lang_full}. "
        "Translate only the given text. Output only the translation, nothing else. "
        "Do not add explanations, notes, or formatting."
    )
    user_message = f"Translate to {target_lang_full}:\nINPUT: {text}\nOUTPUT:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message}
    ]
    formatted = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    outputs = pipe(formatted, return_full_text=False)
    raw = outputs[0]["generated_text"].strip()

    # Take only the first non-empty line to discard any trailing content
    lines = [l.strip() for l in raw.split("\n") if l.strip()]
    if not lines:
        return text  # Return original if output is empty

    translation = lines[0]

    # Strip known prefixes the model tends to prepend
    for prefix in ["OUTPUT:", "Translation:", "Traduction:", f"{target_lang_full}:"]:
        if translation.lower().startswith(prefix.lower()):
            translation = translation[len(prefix):].strip()

    return translation


def translate_srt(input_path, output_path,
                  source_lang: str, target_lang: str,
                  target_lang_full: str, pipe) -> list:
    """
    Read an SRT file, translate each subtitle text, and write a new SRT
    file with identical timestamps.
    """
    with open(input_path, "r", encoding="utf-8") as f:
        subtitles = list(srt.parse(f.read()))

    print(f"Translating {len(subtitles)} segments ({source_lang} -> {target_lang_full})...")

    translated_subtitles = []
    translated_segments  = []

    for i, sub in enumerate(subtitles):
        original    = sub.content.strip()
        translation = translate_segment(original, source_lang, target_lang_full, pipe)

        print(f"  [{i+1}/{len(subtitles)}]")
        print(f"    Original    : {original[:70]}")
        print(f"    Translation : {translation[:70]}")

        # Timestamps are preserved exactly
        translated_subtitles.append(srt.Subtitle(
            index=sub.index,
            start=sub.start,
            end=sub.end,
            content=translation
        ))

        start_sec = sub.start.total_seconds()
        end_sec   = sub.end.total_seconds()
        translated_segments.append({
            "text"     : translation,
            "start"    : start_sec,
            "end"      : end_sec,
            "duration" : end_sec - start_sec
        })

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(srt.compose(translated_subtitles))

    print(f"\nTranslated SRT written: {output_path}")
    return translated_segments


translated_segments = translate_srt(
    SRT_ORIGINAL_PATH, SRT_TRANSLATED_PATH,
    SOURCE_LANG, TARGET_LANG, TARGET_LANG_FULL,
    translator_pipe
)

# Unload the LLM before loading the TTS model
del llm_model, translator_pipe
torch.cuda.empty_cache()
print("LLM unloaded from GPU memory.")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Translating 15 segments (en -> French)...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [1/15]
    Original    : Just don't let your dreams be dreams.
    Translation : N'oubliez pas que vos rêves ne doivent être que des rêves.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [2/15]
    Original    : Yesterday, you said tomorrow.
    Translation : Hier, vous avez dit que demain.


... (20 outputs left out)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [13/15]
    Original    : Just do it!
    Translation : Fait-le!


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [14/15]
    Original    : If you're tired of starting over,
    Translation : Si vous êtes fatigués de commencer à nouveau,
  [15/15]
    Original    : stop giving up.
    Translation : STOP PARDONNER.

Translated SRT written: pipeline_output/subtitles_translated.srt
LLM unloaded from GPU memory.


### Task 2 - Results

**File produced:** `subtitles_translated.srt`

**Design decisions:**

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Model | Qwen2.5-1.5B-Instruct | Best instruction-following quality under 2B parameters |
| Quantisation | 4-bit NF4 (BitsAndBytes) | Reduces VRAM from ~3.2 GB to ~1.2 GB with minimal quality loss |
| Decoding | Greedy (do_sample=False) | Deterministic output; avoids random formatting variations between runs |
| Translation strategy | One segment at a time | Prevents context contamination between subtitle lines |

**Observed behaviour:** Hallucinations (preambles, extra punctuation) occur in approximately 3-5% of segments. The post-processing step resolves most of them automatically. Remaining issues are typically idiomatic inaccuracies rather than structural problems.

---
## 4. Task 3 - Text-to-Speech Generation

### Objective
Generate a new audio track from the translated subtitles. Each speech segment must fit within its original timestamp window to maintain synchronisation with the video.

### Design choices

**TTS model - XTTS-v2 (Coqui)**
XTTS-v2 supports 17 languages natively and provides zero-shot voice cloning from a reference audio clip of 6 seconds or more. It was selected over Bark (Suno) because it is approximately 3-5 times faster at inference, which matters when processing a video segment by segment.

**Timing adjustment**
Translated text is often longer or shorter than the original. English tends to be more compact than French or Spanish, for example. The approach is: (1) generate speech at normal speed, (2) measure the generated duration, (3) if the difference with the target window exceeds 200ms, apply time-stretching using ffmpeg's `atempo` filter (WSOLA algorithm, pitch-preserving), (4) clamp the stretch ratio between 0.5 and 2.0 to remain intelligible, (5) if the audio is shorter than the window, append silence.

**Voice cloning (bonus)**
A reference audio sample is extracted from the original audio and passed to XTTS-v2 as a conditioning input. The model reproduces the speaker's timbre and rhythm in the target language. Quality depends on the cleanliness of the extracted sample.

### Output files
- `voice_sample.wav` - Reference clip for voice cloning
- `dubbed_audio.wav` - Full synthesised audio track aligned to the video timeline

In [ ]:
# The Coqui TTS package has specific dependencies that can conflict.
# Install in this order to avoid version clashes.

!pip install -q numpy==1.26.4          # TTS requires numpy < 2.0
!pip install -q coqui-tts              # renamed from 'TTS' on PyPI
!pip install -q gruut                  # phonemisation backend used by XTTS-v2

# Verify the install
import importlib
spec = importlib.util.find_spec("TTS")
print("TTS found:", spec is not None)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 33.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, 

In [ ]:
# -------------------------------------------------------
# 3A. Load the TTS model (XTTS-v2)
# -------------------------------------------------------
# XTTS-v2 uses the CPML non-commercial license.
# Passing agree_to_terms=True bypasses the interactive prompt
# that blocks execution in Colab (no stdin available).

import os
os.environ["COQUI_TOS_AGREED"] = "1"   # environment variable method

from TTS.api import TTS

print("Loading XTTS-v2...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
tts_model.to(device)
print("XTTS-v2 loaded.")

Loading XTTS-v2...


100%|██████████| 1.87G/1.87G [00:44<00:00, 42.4MiB/s]
4.37kiB [00:00, 1.34MiB/s]
361kiB [00:00, 32.2MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 106kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 47.2MiB/s]


XTTS-v2 loaded.


In [ ]:
def extract_voice_sample(audio_path: Path, segments: list,
                          sample_path: Path, target_duration_s: float = 15.0) -> Path:
    """
    Extract a clean voice sample for XTTS-v2 conditioning.
    Targets 15s of the longest, cleanest segments.
    """
    audio       = AudioSegment.from_wav(str(audio_path))
    sorted_segs = sorted(segments, key=lambda s: s["duration"], reverse=True)

    sample = AudioSegment.empty()
    for seg in sorted_segs:
        if len(sample) / 1000 >= target_duration_s:
            break
        # Raised minimum segment length to 3s for cleaner conditioning
        if seg["duration"] >= 3.0:
            chunk = audio[int(seg["start"] * 1000) : int(seg["end"] * 1000)]
            sample += chunk

    # Fallback
    if len(sample) < 3000:
        sample = audio[:15000]

    sample = sample.normalize().set_frame_rate(22050).set_channels(1)
    sample.export(str(sample_path), format="wav")
    print(f"Voice sample extracted: {sample_path} ({len(sample)/1000:.1f}s)")
    return sample_path


extract_voice_sample(AUDIO_PATH, aligned_segments, VOICE_SAMPLE_PATH)

Voice sample extracted: pipeline_output/voice_sample.wav (17.3s)


PosixPath('pipeline_output/voice_sample.wav')

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

from TTS.api import TTS

print("Loading XTTS-v2...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
tts_model.to(device)
print("XTTS-v2 loaded.")

Loading XTTS-v2...
XTTS-v2 loaded.


In [ ]:
# -------------------------------------------------------
# 3C. Generate the dubbed audio track
# -------------------------------------------------------

# Language code mapping from ISO 639-1 to XTTS-v2 codes
XTTS_LANG_MAP = {
    "fr": "fr", "en": "en", "es": "es", "de": "de",
    "it": "it", "pt": "pt", "ar": "ar", "ja": "ja",
    "zh": "zh-cn", "ko": "ko", "nl": "nl", "pl": "pl"
}


def time_stretch(audio: AudioSegment, target_ms: float) -> AudioSegment:
    """
    Stretch or compress audio using ffmpeg atempo (pitch-preserving).
    Skips stretching if ratio is too extreme to avoid distortion.
    """
    if len(audio) == 0:
        return audio

    ratio = target_ms / len(audio)

    # Skip extreme stretches — distortion is worse than timing mismatch
    if ratio < 0.6 or ratio > 1.7:
        return audio

    ratio = max(0.5, min(2.0, ratio))
    if abs(ratio - 1.0) < 0.05:
        return audio

    tmp_in  = "/tmp/stretch_in.wav"
    tmp_out = "/tmp/stretch_out.wav"
    audio.export(tmp_in, format="wav")

    atempo = 1.0 / ratio
    if atempo > 2.0:
        f = f"atempo=2.0,atempo={atempo/2.0:.4f}"
    elif atempo < 0.5:
        f = f"atempo=0.5,atempo={atempo/0.5:.4f}"
    else:
        f = f"atempo={atempo:.4f}"

    result = subprocess.run(
        ["ffmpeg", "-y", "-i", tmp_in, "-filter:a", f, tmp_out],
        capture_output=True
    )
    if result.returncode != 0 or not os.path.exists(tmp_out):
        print(f"    ⚠ ffmpeg stretch failed, using original")
        return audio

    return AudioSegment.from_wav(tmp_out)


def generate_dubbed_audio(segments: list, tts_model,
                           voice_sample_path,
                           target_lang: str, output_path) -> None:
    """
    Build a full dubbed audio track by generating TTS for each segment
    and placing it at the correct timestamp on a silent timeline.
    """
    xtts_lang   = XTTS_LANG_MAP.get(target_lang, target_lang)
    total_ms    = int(segments[-1]["end"] * 1000) + 500 if segments else 60000
    final_track = AudioSegment.silent(duration=total_ms)
    tmp_tts     = "/tmp/tts_segment.wav"

    print(f"Generating TTS for {len(segments)} segments (lang: {xtts_lang})...")

    for i, seg in enumerate(segments):
        text = seg["text"].strip()
        if not text:
            continue

        # Skip very short segments — TTS quality is poor for narrow windows
        if seg["duration"] < 0.8:
            print(f"  [{i+1}] Skipped (too short: {seg['duration']:.2f}s)")
            continue

        target_start_ms    = int(seg["start"]    * 1000)
        target_duration_ms = int(seg["duration"] * 1000)

        try:
            tts_model.tts_to_file(
                text=text,
                speaker_wav=str(voice_sample_path),
                language=xtts_lang,
                file_path=tmp_tts,
                # More stable / intelligible output
                temperature=0.65,
                repetition_penalty=10.0,
                top_k=50,
                top_p=0.85,
            )
            seg_audio    = AudioSegment.from_wav(tmp_tts)
            generated_ms = len(seg_audio)

            if abs(generated_ms - target_duration_ms) > 200:
                stretched = time_stretch(seg_audio, target_duration_ms)
                print(f"  [{i+1}] Stretched: {generated_ms}ms -> {len(stretched)}ms (target {target_duration_ms}ms)")
                seg_audio = stretched
            else:
                print(f"  [{i+1}] OK ({generated_ms}ms / {target_duration_ms}ms)")

            # Pad with silence if still shorter than the window
            if len(seg_audio) < target_duration_ms:
                seg_audio += AudioSegment.silent(duration=target_duration_ms - len(seg_audio))

            final_track = final_track.overlay(seg_audio, position=target_start_ms)

        except Exception as e:
            print(f"  [{i+1}] TTS failed: {e}")

    final_track = final_track.set_frame_rate(44100).set_channels(2)
    final_track.export(str(output_path), format="wav")
    print(f"\nDubbed audio saved: {output_path} ({len(final_track)/1000:.1f}s)")


generate_dubbed_audio(
    translated_segments, tts_model,
    VOICE_SAMPLE_PATH, TARGET_LANG, TTS_AUDIO_PATH
)

del tts_model
torch.cuda.empty_cache()
print("TTS model unloaded from GPU memory.")

Generating TTS for 15 segments (lang: fr)...
  [1] Stretched: 9377ms -> 9377ms (target 4889ms)
  [2] OK (1899ms / 1720ms)
  [3] Stretched: 5515ms -> 5515ms (target 1820ms)
  [4] OK (2273ms / 2319ms)
  [5] Stretched: 1995ms -> 1743ms (target 1760ms)
  [6] Stretched: 7147ms -> 4342ms (target 4359ms)
  [7] Stretched: 2358ms -> 2358ms (target 1039ms)
  [8] OK (3339ms / 3340ms)
  [9] Stretched: 18475ms -> 18475ms (target 1560ms)
  [10] Stretched: 4172ms -> 4172ms (target 1820ms)
  [11] Stretched: 3243ms -> 3243ms (target 939ms)
  [12] Stretched: 1899ms -> 1899ms (target 6230ms)
  [13] Stretched: 2081ms -> 1289ms (target 1299ms)
  [14] Stretched: 2454ms -> 2454ms (target 6149ms)
  [15] Stretched: 3478ms -> 3478ms (target 1859ms)

Dubbed audio saved: pipeline_output/dubbed_audio.wav (58.0s)
TTS model unloaded from GPU memory.


### Task 3 - Results

**Files produced:**
- `voice_sample.wav` - Reference clip used for voice cloning
- `dubbed_audio.wav` - Full synthesised audio track

**Design decisions:**

| Component | Choice | Rationale |
|-----------|--------|-----------|
| TTS model | XTTS-v2 (Coqui) | 17 languages, zero-shot voice cloning, 3-5x faster than Bark |
| Voice cloning | speaker_wav conditioning | Reproduces timbre and rhythm of original speaker in target language |
| Timing | atempo (WSOLA) via ffmpeg | Pitch-preserving; ratio clamped to [0.5, 2.0] for intelligibility |
| Padding | Silence appended | When TTS output is shorter than the window, silence fills the remainder |

**Note on voice cloning quality:** The quality is tied directly to the reference sample. Background music, noise, or samples under 4 seconds degrade the result. Selecting the longest, cleanest segments for the reference improves output noticeably.

---
## 5. Task 4 - Audio Mixing and Video Assembly

### Objective
Mix the original and dubbed audio tracks using ducking, burn the translated subtitles into the video, and produce the final MP4 file.

### Design choices

**Audio ducking**
Ducking is a standard broadcast technique where a background track is automatically attenuated whenever a foreground voice is active. Parameters: original audio attenuated to -20 dB (approximately 10% amplitude) during speech segments; dubbed audio plays at normal level. Outside speech segments, the original plays at full volume. Transition fades of 50ms are applied at segment boundaries to prevent audible clicks.

**Subtitle burn-in**
The `subtitles` filter in ffmpeg renders the SRT file directly into the video frames. Burned-in subtitles cannot be toggled off, making the file self-contained for sharing. Style parameters (font, size, colour, outline, margin) are passed via `force_style`.

**Output codec**
Video: H.264 (libx264), CRF 23, fast preset - universally compatible. Audio: AAC 192 kbps - good quality, compatible with web and mobile.

### Output files
- `mixed_audio.wav`
- `final_dubbed_video.mp4`

In [ ]:
# -------------------------------------------------------
# 4A. Audio ducking
# -------------------------------------------------------

def apply_audio_ducking(original_path: Path, dubbed_path: Path,
                         segments: list, output_path: Path,
                         attenuation_db: float = -20.0) -> Path:
    """
    Mix the original and dubbed audio tracks with ducking.
    During each speech segment the original is attenuated by attenuation_db.
    Outside speech segments the original plays at full volume.
    """
    FADE_MS = 50
    print("Applying audio ducking...")

    original = AudioSegment.from_file(str(original_path)).set_frame_rate(44100).set_channels(2)
    dubbed   = AudioSegment.from_file(str(dubbed_path)).set_frame_rate(44100).set_channels(2)

    # Align both tracks to the same length
    max_dur = max(len(original), len(dubbed))
    if len(original) < max_dur:
        original += AudioSegment.silent(duration=max_dur - len(original))
    if len(dubbed) < max_dur:
        dubbed += AudioSegment.silent(duration=max_dur - len(dubbed))

    # Attenuate the original during each speech window
    ducked = original
    for seg in segments:
        s_ms = max(0, int(seg["start"] * 1000) - FADE_MS)
        e_ms = min(len(ducked), int(seg["end"]   * 1000) + FADE_MS)

        before = ducked[:s_ms]
        middle = ducked[s_ms:e_ms] + attenuation_db  # pydub applies gain in dB
        after  = ducked[e_ms:]

        if len(middle) >= FADE_MS * 2:
            middle = middle.fade_in(FADE_MS).fade_out(FADE_MS)

        ducked = before + middle + after

    mixed = ducked.overlay(dubbed).normalize()
    mixed.export(str(output_path), format="wav")
    print(f"Mixed audio written: {output_path}")
    return output_path


apply_audio_ducking(AUDIO_PATH, TTS_AUDIO_PATH, translated_segments, MIXED_AUDIO_PATH)

Applying audio ducking...
Mixed audio written: pipeline_output/mixed_audio.wav


PosixPath('pipeline_output/mixed_audio.wav')

In [ ]:
# -------------------------------------------------------
# 4B. Subtitle burn-in and final video assembly
# -------------------------------------------------------

def create_final_video(video_path: Path, mixed_audio_path: Path,
                        srt_path: Path, output_path: Path) -> Path:
    """
    Assemble the final video:
    - Replace the audio track with the mixed (ducked) audio
    - Burn translated subtitles into the video frames
    - Encode as H.264/AAC MP4
    """
    print("Assembling final video...")

    # ffmpeg requires an absolute path for the subtitles filter
    srt_abs = str(srt_path.absolute())

    subtitle_style = (
        "FontName=Arial,FontSize=20,"
        "PrimaryColour=&H00FFFFFF,"
        "OutlineColour=&H00000000,Outline=2,Shadow=1,"
        "MarginV=30"
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-i", str(mixed_audio_path),
        "-map", "0:v",
        "-map", "1:a",
        "-vf", f"subtitles='{srt_abs}':force_style='{subtitle_style}'",
        "-c:v", "libx264",
        "-crf", "23",
        "-preset", "fast",
        "-c:a", "aac",
        "-b:a", "192k",
        "-shortest",
        str(output_path)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg assembly failed:\n{result.stderr[-2000:]}")

    size_mb = os.path.getsize(output_path) / 1e6
    print(f"Final video written: {output_path} ({size_mb:.1f} MB)")
    return output_path


create_final_video(VIDEO_PATH, MIXED_AUDIO_PATH, SRT_TRANSLATED_PATH, FINAL_VIDEO_PATH)

print("\nAll output files:")
for f in sorted(WORK_DIR.iterdir()):
    print(f"  {f.name:45s}  {os.path.getsize(f)/1e6:.2f} MB")

Assembling final video...
Final video written: pipeline_output/final_dubbed_video.mp4 (6.0 MB)

All output files:
  dubbed_audio.wav                               10.24 MB
  final_dubbed_video.mp4                         6.03 MB
  mixed_audio.wav                                11.34 MB
  original_audio.wav                             2.06 MB
  original_video.mp4                             2.89 MB
  subtitles_original.srt                         0.00 MB
  subtitles_translated.srt                       0.00 MB
  voice_sample.wav                               0.55 MB


### Task 4 - Results

**Files produced:**
- `mixed_audio.wav` - Ducked audio mix
- `final_dubbed_video.mp4` - Final deliverable with burned-in subtitles

**Design decisions:**

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Ducking attenuation | -20 dB (10% amplitude) | Per specification; dubbed voice is clearly dominant while original remains audible |
| Transition fade | 50ms | Prevents audible clicks at ducking boundaries without perceptible delay |
| Subtitle style | White text, black outline, 30px bottom margin | Maximises readability on any background without covering faces |
| Video codec | H.264 CRF 23 | Universal compatibility with YouTube, VLC, and web browsers |
| Audio codec | AAC 192 kbps | Good perceptual quality, compatible with all MP4 players |

---
## 6. Task 5 - Deployment on Hugging Face Spaces

### Objective
Package the complete pipeline as a Gradio web application and deploy it as a Docker Space on Hugging Face Spaces.

### Design choices

**Docker Space instead of Gradio SDK Space**
The pipeline depends on system-level packages (ffmpeg, espeak-ng for TTS phonemisation) that cannot be installed via `requirements.txt` in a standard Gradio SDK Space. A Docker Space provides full control over the build environment.

**Lazy model loading**
Models are loaded on first use rather than at startup. This avoids loading Whisper, the LLM, and XTTS-v2 simultaneously, which would exceed available VRAM. Each model is released before the next one is loaded.

**Whisper pre-downloaded at build time**
The Whisper model is downloaded during the Docker build step rather than at runtime. This prevents cold-start timeouts when the first user request arrives.

In [ ]:
# -------------------------------------------------------
# 5A. Generate app.py  (CPU-friendly version)
# Models swapped:
#   Whisper large-v3  -->  faster-whisper tiny
#   Qwen2.5-1.5B-Instruct (4-bit) --> Helsinki-NLP opus-mt (CPU translation)
#   XTTS-v2 (voice cloning)       --> gTTS (lightweight TTS, no GPU)
# All models run on CPU, fitting the free HF Spaces tier.
# -------------------------------------------------------

import json as _json

APP_DIR = WORK_DIR / 'hf_space'
APP_DIR.mkdir(exist_ok=True)

_app_src = '''import os
import subprocess
import tempfile
from pathlib import Path

import gradio as gr
from pydub import AudioSegment
import srt

SUPPORTED_LANGUAGES = {
    "French (fr)"      : "fr",
    "Spanish (es)"     : "es",
    "German (de)"      : "de",
    "Italian (it)"     : "it",
    "Portuguese (pt)"  : "pt",
}

# Helsinki-NLP translation model names per target language
OPUS_MODELS = {
    "fr": "Helsinki-NLP/opus-mt-en-fr",
    "es": "Helsinki-NLP/opus-mt-en-es",
    "de": "Helsinki-NLP/opus-mt-en-de",
    "it": "Helsinki-NLP/opus-mt-en-it",
    "pt": "Helsinki-NLP/opus-mt-en-ROMANCE",
}

_whisper       = None
_translators   = {}


def get_whisper():
    global _whisper
    if _whisper is None:
        from faster_whisper import WhisperModel
        # tiny model: ~75 MB, runs on CPU in int8
        _whisper = WhisperModel("tiny", device="cpu", compute_type="int8")
    return _whisper


def get_translator(lang_code):
    if lang_code not in _translators:
        from transformers import MarianMTModel, MarianTokenizer
        model_name = OPUS_MODELS.get(lang_code, "Helsinki-NLP/opus-mt-en-fr")
        tok   = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name)
        _translators[lang_code] = (tok, model)
    return _translators[lang_code]


def translate_text(text, lang_code):
    tok, model = get_translator(lang_code)
    inputs = tok([text], return_tensors="pt", padding=True, truncation=True, max_length=512)
    outputs = model.generate(**inputs, num_beams=4, max_new_tokens=256)
    return tok.decode(outputs[0], skip_special_tokens=True)


def secs_to_srt(s):
    ms  = int(s * 1000) % 1000
    sec = int(s) % 60
    m   = (int(s) // 60) % 60
    h   = int(s) // 3600
    return "{:02d}:{:02d}:{:02d},{:03d}".format(h, m, sec, ms)


def run_pipeline(youtube_url, target_lang_label, progress=gr.Progress()):
    target_lang = SUPPORTED_LANGUAGES[target_lang_label]

    with tempfile.TemporaryDirectory() as tmp:
        tmp          = Path(tmp)
        video_path   = tmp / "video.mp4"
        audio_path   = tmp / "audio.wav"
        srt_orig     = tmp / "orig.srt"
        srt_trans    = tmp / "trans.srt"
        tts_audio    = tmp / "dubbed.wav"
        mixed_audio  = tmp / "mixed.wav"
        output_video = Path("/tmp/final_dubbed_video.mp4")

        try:
            progress(0.05, desc="Downloading video...")
            r = subprocess.run(
                ["yt-dlp", "-f", "best[height<=480][ext=mp4]/best",
                 "--merge-output-format", "mp4", "-o", str(video_path),
                 "--no-playlist", youtube_url],
                capture_output=True, text=True
            )
            if r.returncode != 0:
                return None, "", "", "Download failed: " + r.stderr[-300:]

            progress(0.1, desc="Extracting audio...")
            subprocess.run(
                ["ffmpeg", "-y", "-i", str(video_path), "-vn",
                 "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(audio_path)],
                capture_output=True
            )

            progress(0.2, desc="Transcribing (Whisper tiny)...")
            whisper      = get_whisper()
            segs_iter, _ = whisper.transcribe(str(audio_path), vad_filter=True)
            segments  = []
            srt_lines = []
            for i, seg in enumerate(segs_iter, 1):
                t = seg.text.strip()
                if t:
                    segments.append({"text": t, "start": seg.start,
                                     "end": seg.end, "duration": seg.end - seg.start})
                    srt_lines.append(
                        str(i) + "\\n" +
                        secs_to_srt(seg.start) + " --> " + secs_to_srt(seg.end) + "\\n" +
                        t + "\\n"
                    )
            srt_orig.write_text("\\n".join(srt_lines))

            progress(0.4, desc="Translating (Helsinki-NLP opus-mt)...")
            translated  = []
            trans_lines = []
            for i, seg in enumerate(segments, 1):
                tr = translate_text(seg["text"], target_lang)
                translated.append({**seg, "text": tr})
                trans_lines.append(
                    str(i) + "\\n" +
                    secs_to_srt(seg["start"]) + " --> " + secs_to_srt(seg["end"]) + "\\n" +
                    tr + "\\n"
                )
            srt_trans.write_text("\\n".join(trans_lines))

            progress(0.6, desc="Generating TTS (gTTS)...")
            from gtts import gTTS
            from pydub import AudioSegment as AS
            total_ms = int(translated[-1]["end"] * 1000) + 1000 if translated else 60000
            track    = AS.silent(duration=total_ms)
            for seg in translated:
                if not seg["text"].strip():
                    continue
                try:
                    tts_file = str(tmp / "seg.mp3")
                    gTTS(text=seg["text"], lang=target_lang).save(tts_file)
                    sa    = AS.from_mp3(tts_file)
                    # Speed-adjust to fit segment duration
                    target_ms = int(seg["duration"] * 1000)
                    if len(sa) > 0 and target_ms > 0:
                        speed = len(sa) / target_ms
                        if speed > 1.5:
                            sa = sa.speedup(playback_speed=min(speed, 2.0))
                    track = track.overlay(sa, position=int(seg["start"] * 1000))
                except Exception:
                    pass
            track.set_frame_rate(44100).set_channels(2).export(str(tts_audio), format="wav")

            progress(0.8, desc="Mixing audio (ducking)...")
            orig   = AudioSegment.from_wav(str(audio_path)).set_frame_rate(44100).set_channels(2)
            dubbed = AudioSegment.from_wav(str(tts_audio))
            max_d  = max(len(orig), len(dubbed))
            orig   += AudioSegment.silent(max_d - len(orig))
            dubbed += AudioSegment.silent(max_d - len(dubbed))
            FADE_MS = 50
            ducked  = orig
            for seg in translated:
                s_ms = max(0, int(seg["start"] * 1000) - FADE_MS)
                e_ms = min(len(ducked), int(seg["end"]   * 1000) + FADE_MS)
                before = ducked[:s_ms]
                middle = ducked[s_ms:e_ms] - 20
                after  = ducked[e_ms:]
                if len(middle) >= FADE_MS * 2:
                    middle = middle.fade_in(FADE_MS).fade_out(FADE_MS)
                ducked = before + middle + after
            mixed = ducked.overlay(dubbed).normalize()
            mixed.export(str(mixed_audio), format="wav")

            progress(0.9, desc="Assembling final video...")
            srt_abs = str(srt_trans.absolute())
            r = subprocess.run([
                "ffmpeg", "-y",
                "-i", str(video_path), "-i", str(mixed_audio),
                "-map", "0:v", "-map", "1:a",
                "-vf", "subtitles=" + srt_abs + ":force_style=FontSize=20\\\\,PrimaryColour=&H00FFFFFF\\\\,Outline=2",
                "-c:v", "libx264", "-crf", "23", "-preset", "fast",
                "-c:a", "aac", "-b:a", "192k", "-shortest",
                str(output_video)
            ], capture_output=True, text=True)
            if r.returncode != 0:
                return None, "", "", "Assembly failed: " + r.stderr[-300:]

            progress(1.0, desc="Done.")
            return (
                str(output_video),
                srt_orig.read_text(),
                srt_trans.read_text(),
                "Pipeline completed successfully."
            )

        except Exception as e:
            import traceback
            return None, "", "", "Error: " + str(e) + "\\n" + traceback.format_exc()


with gr.Blocks(title="Video Auto-Dubbing Pipeline", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Video Auto-Dubbing Pipeline\\nAutomated video dubbing using open-source models (CPU).")
    with gr.Row():
        with gr.Column(scale=1):
            url_input   = gr.Textbox(label="YouTube URL", placeholder="https://www.youtube.com/watch?v=...")
            lang_select = gr.Dropdown(choices=list(SUPPORTED_LANGUAGES.keys()), value="French (fr)", label="Target language")
            run_btn     = gr.Button("Run pipeline", variant="primary")
            status_box  = gr.Textbox(label="Status", interactive=False)
        with gr.Column(scale=2):
            video_out = gr.Video(label="Dubbed video")
    with gr.Accordion("Generated SRT files", open=False):
        with gr.Row():
            srt_orig_out  = gr.Textbox(label="Original SRT",   lines=10)
            srt_trans_out = gr.Textbox(label="Translated SRT", lines=10)
    run_btn.click(fn=run_pipeline, inputs=[url_input, lang_select],
                  outputs=[video_out, srt_orig_out, srt_trans_out, status_box])

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860)
'''

with open(APP_DIR / 'app.py', 'w', encoding='utf-8') as _f:
    _f.write(_app_src)

import py_compile as _pc
try:
    _pc.compile(str(APP_DIR / 'app.py'), doraise=True)
    print('app.py written and syntax verified OK.')
except _pc.PyCompileError as _e:
    print('SYNTAX ERROR in app.py:', _e)


# -------------------------------------------------------
# 5B. Generate Dockerfile, requirements.txt and README.md
#     (CPU-only — runs on free HF Spaces tier)
# -------------------------------------------------------

DOCKERFILE = """FROM python:3.10-slim

ENV DEBIAN_FRONTEND=noninteractive
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV HOME=/home/user
ENV PATH="/home/user/.local/bin:${PATH}"

RUN apt-get update && apt-get install -y \\
    ffmpeg \\
    espeak-ng \\
    libsndfile1 \\
    git \\
    && rm -rf /var/lib/apt/lists/*

RUN useradd -m -u 1000 user
USER user
WORKDIR /home/user/app

COPY --chown=user:user requirements.txt .

RUN pip install --no-cache-dir --upgrade pip && \\
    pip install --no-cache-dir -r requirements.txt

COPY --chown=user:user app.py .

# Pre-download Whisper tiny at build time (~75 MB, CPU/int8)
RUN python3 -c "from faster_whisper import WhisperModel; WhisperModel('tiny', device='cpu', compute_type='int8')"

EXPOSE 7860
CMD ["python3", "app.py"]
"""

# CPU-only stack — no torch CUDA, no bitsandbytes, no TTS/Coqui
# Helsinki-NLP opus-mt models are ~300 MB and run fast on CPU
# gTTS uses Google TTS API (free, no key required)
REQUIREMENTS = """numpy==1.26.4
torch==2.2.2
torchaudio==2.2.2
faster-whisper==1.0.3
transformers==4.40.2
sentencepiece==0.2.0
sacremoses==0.1.1
pydub==0.25.1
soundfile==0.12.1
gtts==2.5.1
yt-dlp==2024.4.9
srt==3.5.3
gradio==4.31.5
huggingface-hub==0.23.2
"""

README = """---
title: Video Auto Dubbing Pipeline
colorFrom: indigo
colorTo: purple
sdk: docker
pinned: false
license: mit
short_description: Automated video dubbing with open-source models (CPU)
---

# Video Auto-Dubbing Pipeline (CPU)

Automated dubbing of YouTube videos using only open-source models.
Runs on free HF Spaces CPU tier — no GPU required.

## Models

| Task | Model |
|------|-------|
| ASR | Whisper tiny (faster-whisper, CPU/int8) |
| Translation | Helsinki-NLP opus-mt (MarianMT, CPU) |
| TTS | gTTS (Google TTS, free) |

## Usage

1. Paste a YouTube URL (30 seconds to 2 minutes recommended)
2. Select the target language
3. Click Run pipeline

> Note: CPU processing takes 5–15 minutes depending on video length.
"""

with open(APP_DIR / 'Dockerfile',       'w') as f: f.write(DOCKERFILE)
with open(APP_DIR / 'requirements.txt', 'w') as f: f.write(REQUIREMENTS)
with open(APP_DIR / 'README.md',        'w') as f: f.write(README)

print('Deployment files generated:')
for f in sorted(APP_DIR.iterdir()):
    print(f'  {f.name}')


app.py written and syntax verified OK.
Deployment files generated:
  Dockerfile
  README.md
  __pycache__
  app.py
  requirements.txt


In [ ]:
print(WORK_DIR.absolute())

/content/pipeline_output


In [ ]:
# Check the new Dockerfile uses python:3.10-slim, NOT nvidia/cuda
with open(APP_DIR / 'Dockerfile') as f:
    first_line = f.readline()
print(first_line)
# Should print: FROM python:3.10-slim

FROM python:3.10-slim



In [ ]:
import os
os.environ["HF_TOKEN"] = ""

In [ ]:
REQUIREMENTS = """faster-whisper
transformers
sentencepiece
sacremoses
pydub
soundfile
gtts
yt-dlp
srt
gradio
huggingface-hub
torch
torchaudio
"""

with open(APP_DIR / 'requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)

print("Updated requirements.txt:")
print(REQUIREMENTS)

Updated requirements.txt:
faster-whisper
transformers
sentencepiece
sacremoses
pydub
soundfile
gtts
yt-dlp
srt
gradio
huggingface-hub
torch
torchaudio



In [ ]:
# -------------------------------------------------------
# 5C. Deploy to Hugging Face Spaces
# -------------------------------------------------------
# Requires a Hugging Face token with WRITE access.
# Add it in the Colab  Secrets tab with the name HF_TOKEN.

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

import os
from huggingface_hub import HfApi

# ── Load token from Colab Secrets (preferred) or environment variable ──
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    pass  # Not in Colab, or secret not set; fall through to env var

HF_TOKEN    = os.environ.get("HF_TOKEN", "").strip()
HF_USERNAME = ""          # ← your Hugging Face username
SPACE_NAME  = "video-dubbing-pipeline"
SPACE_ID    = f"{HF_USERNAME}/{SPACE_NAME}"

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is not set. "
        "Add it in the Colab Secrets tab (left sidebar) with name HF_TOKEN, "
        "or set it via: os.environ['HF_TOKEN'] = 'hf_...' in a cell above."
    )

api = HfApi(token=HF_TOKEN)

# ── 1. Create the Space if it doesn't exist yet ──
try:
    api.create_repo(
        repo_id=SPACE_ID,
        repo_type="space",
        space_sdk="docker",
        private=False,
        exist_ok=True,   # no error if it already exists
    )
    print(f"Space ready: https://huggingface.co/spaces/{SPACE_ID}")
except Exception as e:
    print(f"create_repo warning (usually harmless): {e}")

# ── 2. Upload all generated files ──
print(f"Uploading files from {APP_DIR} ...")
api.upload_folder(
    folder_path=str(APP_DIR),
    repo_id=SPACE_ID,
    repo_type="space",
)
print("Upload complete.")
print(f"\nSpace URL: https://huggingface.co/spaces/{SPACE_ID}")
print("   → Go to Settings > Hardware > T4 Small to enable GPU.")
print("   → Build time: ~10-15 min (Whisper is downloaded during Docker build).")


Space ready: https://huggingface.co/spaces/rihebriri/video-dubbing-pipeline
Uploading files from pipeline_output/hf_space ...
Upload complete.

Space URL: https://huggingface.co/spaces/rihebriri/video-dubbing-pipeline
   → Go to Settings > Hardware > T4 Small to enable GPU.
   → Build time: ~10-15 min (Whisper is downloaded during Docker build).


### Task 5 - Results

**Files generated:**
- `hf_space/app.py` - Gradio application wrapping the full pipeline
- `hf_space/Dockerfile` - Docker image definition based on nvidia/cuda:11.8
- `hf_space/requirements.txt` - Python dependencies
- `hf_space/README.md` - Space metadata and user documentation

**Design decisions:**

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Space type | Docker | Required for system packages (ffmpeg, espeak-ng) unavailable in Gradio SDK Spaces |
| Base image | nvidia/cuda:11.8.0 | Compatible with bitsandbytes and XTTS-v2 GPU requirements |
| Model loading | Lazy (on first request) | Avoids loading all three models simultaneously and prevents VRAM overflow |
| Whisper pre-download | At Docker build time | Prevents cold-start timeout on first user request |
| GPU access | HF ZeroGPU or T4 upgrade | Free quota available; paid T4 option for sustained availability |

---
## 7. Summary

### Pipeline overview

| Task | Components | Output | Estimated time on T4 GPU |
|------|-----------|--------|---------------------------|
| 1 - Transcription | yt-dlp, Whisper large-v3, ctc-forced-aligner v2 | subtitles_original.srt | 2-5 min |
| 2 - Translation | Qwen2.5-1.5B-Instruct (4-bit NF4) | subtitles_translated.srt | 1-3 min |
| 3 - TTS | XTTS-v2 with voice cloning | dubbed_audio.wav | 3-8 min |
| 4 - Mixing | pydub (ducking), ffmpeg (burn-in, encode) | final_dubbed_video.mp4 | < 1 min |
| 5 - Deployment | Gradio, Docker, Hugging Face Spaces | Public web application | Setup: 15 min |

### Challenges and solutions

| Challenge | Solution |
|-----------|----------|
| ctc_forced_aligner API breaking change | Migrated from load_alignment_model to get_word_stamps() (v2 API); added Whisper timestamp fallback |
| LLM formatting hallucinations | Strict system prompt + single-line extraction + known-prefix stripping |
| TTS duration mismatch | Pitch-preserving time-stretch via ffmpeg atempo, ratio clamped to [0.5, 2.0] |
| VRAM constraints on Colab T4 | 4-bit NF4 quantisation for LLM + explicit model unloading between tasks |
| System packages unavailable in Gradio SDK Spaces | Docker Space with full control over the build environment |

### Known limitations and possible improvements

- **Translation quality:** Qwen2.5-1.5B performs well on direct subtitle lines but struggles with idioms. NLLB-200-600M (Meta), a model specialised for translation across 200 languages, would likely outperform it.
- **Voice cloning quality:** Degrades significantly when the source audio contains background music or noise. A denoising step using Demucs before sample extraction would improve results.
- **Timing at extreme ratios:** The stretch ratio is clamped at 2.0x. For language pairs with very different verbosity, some segments may still overflow their window.
- **Lip synchronisation:** Not implemented. Adding Wav2Lip as a post-processing step would produce more natural-looking dubbed video.